In [ ]:
# TIME SERIES

In [17]:
# # resample() resample() is like groupby() but for time — it buckets your data into time windows and aggregates.
# resample() is specifically a time-based method. It only works when your index is a datetime.
# resample("h")   # bucket by hour
# resample("D")   # bucket by day
# resample("W")   # bucket by week
# resample("ME")  # bucket by month end
# resample("2h")   # every 2 hours
# resample("15min") # every 15 minutes
# resample("3D")   # every 3 days
# Without a datetime index — resample() throws an error. 
# That's why set_index("timestamp") is always the first step.

In [14]:
import pandas as pd
import numpy as np

df_metrics = pd.read_csv("server_metrics.csv")
df_tickets = pd.read_csv("incidents.csv")
df_logs = pd.read_csv("app_logs.csv")
print(df_metrics.dtypes)
df_metrics["timestamp"] = pd.to_datetime(df_metrics["timestamp"])
print(df_metrics.dtypes)
print(df_metrics)

timestamp       object
server_id       object
cpu_pct        float64
memory_pct     float64
response_ms    float64
disk_pct       float64
status          object
dtype: object
timestamp      datetime64[ns]
server_id              object
cpu_pct               float64
memory_pct            float64
response_ms           float64
disk_pct              float64
status                 object
dtype: object
              timestamp server_id  cpu_pct  memory_pct  response_ms  disk_pct  \
0   2026-01-01 00:00:00    srv-01     49.6        94.6        891.8      68.9   
1   2026-01-01 00:00:00    srv-02     32.3        33.9       1046.1      69.1   
2   2026-01-01 00:00:00    srv-03     21.6        96.0       1007.3      43.8   
3   2026-01-01 00:00:00    srv-04     34.5        50.7        653.5      58.1   
4   2026-01-01 00:00:00    srv-05     68.3        39.5        386.0      53.8   
..                  ...       ...      ...         ...          ...       ...   
520 2026-01-01 04:00:00    srv-01 

In [13]:
# Hourly average CPU across fleet
hourly_cpu = df_metrics.set_index("timestamp").resample("h")["cpu_pct"].mean().round(2)
print(hourly_cpu.head(10))
# gives you fleet-wide average per hour — all 5 servers collapsed into one number per hour.

timestamp
2026-01-01 00:00:00    41.26
2026-01-01 01:00:00    67.16
2026-01-01 02:00:00    76.80
2026-01-01 03:00:00    59.98
2026-01-01 04:00:00    55.54
2026-01-01 05:00:00    54.22
2026-01-01 06:00:00    67.90
2026-01-01 07:00:00    45.56
2026-01-01 08:00:00    53.08
2026-01-01 09:00:00    77.36
Freq: h, Name: cpu_pct, dtype: float64


In [16]:
# Daily max response time per server
daily_response = df_metrics.set_index("timestamp").groupby("server_id", observed=True).resample("D") \
["response_ms"].max().round(2)
print(daily_response.head(5))

server_id  timestamp 
srv-01     2026-01-01    1130.4
           2026-01-02    1129.9
           2026-01-03    1196.2
           2026-01-04    1178.5
           2026-01-05    1149.3
Name: response_ms, dtype: float64


In [22]:
# Daily ticket count
df_tickets["created_at"] = pd.to_datetime(df_tickets["created_at"])
daily_tickets = df_tickets.set_index("created_at").resample("D")["ticket_id"].count()
print(daily_tickets)

created_at
2026-01-01    1
2026-01-02    2
2026-01-03    3
2026-01-04    6
2026-01-05    2
             ..
2026-04-06    1
2026-04-07    2
2026-04-08    3
2026-04-09    2
2026-04-10    3
Freq: D, Name: ticket_id, Length: 100, dtype: int64


In [ ]:
# shift() - compare current vs previous reading

In [23]:
# How much did CPU change from previous reading ?
df_metrics_ts = df_metrics.set_index("timestamp").sort_index()
df_metrics_ts["cpu_prev"]